## Step 1: Import required PySpark libraries

This cell imports the core PySpark modules:
- `SparkSession` for creating and managing the Spark session.
- `functions as F` for SQL-style data transformations.
- `Window` for performing partitioned ranking and window operations.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Step 2: Initialize Spark Session and Configure BigQuery Connector

This cell:
- Creates a new Spark session for BigQuery analysis.
- Adds the BigQuery connector JAR package (`spark-bigquery-with-dependencies`).
- Configures Spark to use your GCP project and dataset for temporary materialization.
- Enables reading views and sets the GCS temporary bucket for intermediate data.

In [ ]:
spark = (SparkSession.builder
         .appName("GoogleTrends_PySpark_Analysis")
         .config("spark.jars.packages",
                 "com.google.cloud.spark:spark-bigquery-with-dependencies_2.12:0.42.1")
         .getOrCreate())

spark.conf.set("viewsEnabled", "true")
spark.conf.set("materializationDataset", "my_temp_dataset")
spark.conf.set("parentProject", "lateral-layout-474104-p8")

## Step 3: Fetch the most recent available `refresh_date` values

Queries the public BigQuery dataset `bigquery-public-data.google_trends.international_top_rising_terms`
to get the **10 most recent refresh dates**.  
These dates help identify valid partitions since the dataset doesn’t always include today’s date.
The latest date (`END_DATE`) will be used for subsequent analysis.

In [ ]:
q_dates = """
SELECT DISTINCT refresh_date
FROM `bigquery-public-data.google_trends.international_top_rising_terms`
ORDER BY refresh_date DESC
LIMIT 10
"""
recent_dates_df = (spark.read.format("bigquery")
                   .option("query", q_dates)
                   .load())
recent_dates = [r.refresh_date.strftime("%Y-%m-%d") for r in recent_dates_df.collect()]
END_DATE = recent_dates[0]

## Step 4: Load Google Trends data for the latest 7-day window

Uses the latest available date (`END_DATE`) to load one week of data using a BigQuery SQL query.
This ensures multiple countries are represented since each `refresh_date` may contain different countries.

Expected result: ~10–20K rows covering trending search terms across various countries.

In [ ]:
query = f"""
SELECT *
FROM `bigquery-public-data.google_trends.international_top_rising_terms`
WHERE refresh_date BETWEEN DATE_SUB("{END_DATE}", INTERVAL 7 DAY) AND "{END_DATE}"
"""
df = (spark.read.format("bigquery")
       .option("query", query)
       .load())

## Step 5: Compute per-country summary statistics

Groups the dataset by `country_code` and `country_name` to calculate:
- Number of rising terms per country (`num_rising_terms`)
- Average search score (`avg_score`)
- Average percent gain (`avg_percent_gain`)

This helps identify which countries have the most trending search activity.

In [ ]:
country_summary = (df.groupBy("country_code", "country_name")
                     .agg(F.count("*").alias("num_rising_terms"),
                          F.avg("score").alias("avg_score"),
                          F.avg("percent_gain").alias("avg_percent_gain")))

## Step 6: Identify the Top 10 most active countries

Ranks countries by their `num_rising_terms` using a window function.
Selects the **top 10 countries** that had the highest number of rising search queries.
These countries will be used for detailed term-level analysis.

In [ ]:
w_countries = Window.orderBy(F.desc("num_rising_terms"))
top_countries_df = (country_summary
                    .withColumn("country_rank", F.dense_rank().over(w_countries))
                    .filter(F.col("country_rank") <= 10)
                    .select("country_code", "country_name"))

## Step 7: Filter original data for the top 10 countries

Joins the full dataset with the `top_countries_df` table to retain only rows belonging to
the top 10 most active countries.

This reduces data size while focusing analysis on the most relevant countries.

In [ ]:
df_top = df.join(top_countries_df, ["country_code", "country_name"], "inner")

## Step 8: Aggregate metrics per (country, term)

Groups the filtered dataset (`df_top`) by **country** and **term**.  
Calculates two averages for each term within a country:  
- `avg_score` – the average popularity score  
- `avg_percent_gain` – the average percentage increase  

Creates a summarized DataFrame named **`agg`** for later ranking.

In [ ]:
agg = (df_top.groupBy("country_code", "country_name", "term")
             .agg(F.avg("score").alias("avg_score"),
                  F.avg("percent_gain").alias("avg_percent_gain")))

## Step 9: Rank top N terms per country

Ranks each term within its country based on its average score and percent gain.  
Keeps the **top 5 terms** per country using a window function.  
Generates a new DataFrame **`top_terms_ranked`** with rank numbers (1 to 5).

In [ ]:
w_terms = Window.partitionBy("country_code").orderBy(F.desc("avg_score"),
                                                     F.desc("avg_percent_gain"),
                                                     F.asc("term"))
top_terms_ranked = (agg.withColumn("rank", F.row_number().over(w_terms))
                      .filter(F.col("rank") <= 5)
                      .orderBy("country_code", "rank"))

## Step 10: Display top terms per country

Shows the final ranked list of the top terms for each country.  
Helps verify that each country lists up to 5 terms ordered by rank.  
Use this output to observe trending or recurring terms.

In [ ]:
top_terms_ranked.show(200, truncate=False)

+------------+------------+-------------------------------------------------------------+-----------------+------------------+----+
|country_code|country_name|term                                                         |avg_score        |avg_percent_gain  |rank|
+------------+------------+-------------------------------------------------------------+-----------------+------------------+----+
|CO          |Colombia    |irlanda del norte alemania                                   |85.32692307692308|1950.0            |1   |
|CO          |Colombia    |slothobicuan79.online                                        |83.86792452830188|2100.0            |2   |
|CO          |Colombia    |corea del sur paraguay                                       |83.08163265306122|1450.0            |3   |
|CO          |Colombia    |selección femenina de fútbol sub-17 de españa colombia sub-17|82.92            |950.0             |4   |
|CO          |Colombia    |situsbelutjp88.online                            

## Step 11: Save results to BigQuery

Writes the final DataFrame `top_terms_ranked` to your BigQuery dataset.  
Replaces the table if it already exists (overwrite mode).  
If you encounter a GCS error, add `.option("writeMethod","direct")` to the write step.

In [ ]:
top_terms_ranked.write \
    .format("bigquery") \
    .option("table", "lateral-layout-474104-p8.data_228_hw3_4.top_terms_ranked") \
    .option("writeMethod", "direct") \
    .mode("overwrite") \
    .save()